# Mixed-effects model: did water quality change after dam removal?

Reads the outputs of `analysis/mixed_model.py` (run it first: `sbatch data/slurm/fit_mixed_model.slurm` on the cluster, or locally `python analysis/mixed_model.py --case-study all_removals --input analysis_measurement_all_removals.csv`). The notebook doesn't fit anything itself, so it opens in seconds.

## The model in plain English

**The question:** after a dam is removed, does the water downstream change, *beyond what would have changed anyway*?

**The data going in** (same rules as `analysis_robustness.ipynb`):
- **treated** stations: downstream of the dam, and this dam is the measurement's only (or clearly nearest) upstream removal;
- **control** stations: upstream of the same dam, not affected by another removal, and not in the former reservoir;
- one value per station per day (the median), so a logger reading every 15 minutes doesn't count as hundreds of samples;
- skewed parameters (turbidity, conductance, nutrients, TDS) on a log10 scale, so effects read as **% change**.

**Two models per parameter:**

| Model | Uses | Formula | The effect is… |
|---|---|---|---|
| **before/after** | treated stations only | `y ~ post + month + air temp + prior-month rain` | `post`: the average change downstream after removal |
| **BACI** (before-after-control-impact) | dams with both treated *and* control stations | `y ~ post + downstream + post×downstream + month + air temp + prior-month rain` | `post×downstream`: the downstream change **minus** the upstream change at the same dam |

BACI is the stronger design: if a drought or a new treatment plant changes the whole river, upstream changes too and BACI subtracts it out. Before/after has ~5× more dams but can't tell a removal from a regional trend.

**What "mixed effects" adds (the random effects):** every dam gets its *own* baseline level and its *own* removal effect, and every station its own baseline. The headline number is the **average effect across dams**, and its uncertainty reflects how much dams *disagree*, not how many samples there are. That's the fix for the earlier preview models in `analysis_eda`, whose intervals were too narrow because 10,000 samples from 100 dams were treated as 10,000 independent pieces of evidence.

`month`, air temperature and prior-month rain soak up seasonal and weather differences between the before and after samples.

In [ ]:
from pathlib import Path
import json
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from analysis import mixed_model as mm

DATA_ROOT = Path(os.environ.get("DAM_REMOVAL_PROJECT_ROOT", ROOT))
CASE_STUDY = "all_removals"
RESULTS_DIR = mm.output_dir_for(CASE_STUDY, DATA_ROOT)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

MODEL_LABELS = {"baci": "BACI (vs upstream)", "before_after": "before/after (downstream)"}
MODEL_STYLE = {"baci": {"color": "#2a78d6", "marker": "o"},
               "before_after": {"color": "#eb6834", "marker": "s"}}
CHECK_SERIES = "dissolved_oxygen"   # series whose residuals section 4 plots

if not (RESULTS_DIR / "results.csv").exists():
    raise FileNotFoundError(f"No model outputs in {RESULTS_DIR}: run analysis/mixed_model.py first.")
results = pd.read_csv(RESULTS_DIR / "results.csv")
dam_effects = pd.read_csv(RESULTS_DIR / "dam_effects.csv")
residuals = pd.read_parquet(RESULTS_DIR / "residuals.parquet")
settings = json.loads((RESULTS_DIR / "settings.json").read_text())
SERIES = [s for s in settings["series"] if s in set(results["series"])]

print(f"Input: {settings['input']}")
for model, formula in settings["formulas"].items():
    print(f"{model:>13}: {formula}   (+ random effects by dam and station)")

## 1. The headline table

One row per parameter and model. Read it like this:
- **effect**: the average change across dams, with its 95% interval. For log-scale parameters it's a % change (+20% = 1.2× higher after removal); otherwise it's in the parameter's unit.
- **verdict**: `higher after` / `lower after` when the whole interval is on one side of 0; otherwise `no clear change`. "No clear change" doesn't mean "no change": with few dams the interval can be wide.
- **dams**: how many dams the estimate rests on. Below ~15, treat it as a hint.
- **fit_check**: `check` means the optimizer flagged the fit (usually harmless, see section 4); compare that row with the dam-bootstrap estimate in `analysis_robustness.ipynb`.

With 8 parameters × 2 models, about 1 interval in 20 excludes 0 by chance alone, so a single "higher/lower" that isn't backed by the other model or by `analysis_robustness` is weak evidence.

In [ ]:
def readable(row):
    # The effect and its interval as text, in % for log-scale series.
    if pd.isna(row.get("estimate")):
        return "not fit: " + str(row.get("error", ""))
    if row["log_scale"]:
        return f"{row['pct_estimate']:+.1f}% [{row['pct_ci_low']:+.1f}, {row['pct_ci_high']:+.1f}]"
    unit = row["unit"]
    return f"{row['estimate']:+.3f} [{row['ci_low']:+.3f}, {row['ci_high']:+.3f}] {unit}"


def verdict(row):
    if pd.isna(row.get("estimate")):
        return ""
    if row["ci_low"] > 0:
        return "higher after"
    if row["ci_high"] < 0:
        return "lower after"
    return "no clear change"


table = results.assign(effect=results.apply(readable, axis=1),
                       verdict=results.apply(verdict, axis=1),
                       model=results["model"].map(MODEL_LABELS))
table = table[["series", "model", "effect", "verdict", "p_value", "dams", "stations", "days", "fit_check"]]
table.round({"p_value": 3}).set_index(["series", "model"])

## 2. All effects at a glance

Each panel is one parameter, in its own unit (log-scale ones in %). A dot is the average effect across dams, the bar its 95% interval, and the gray line is "no change". **An interval crossing the gray line is no clear evidence of a change.** Where both models agree in direction, the result is more believable.

In [ ]:
def to_plot_scale(row, key):
    return row[f"pct_{key}"] if row["log_scale"] else row[key]


fig, axes = plt.subplots(2, -(-len(SERIES) // 2), figsize=(4 * -(-len(SERIES) // 2), 5.2), squeeze=False)
for ax in axes.flat[len(SERIES):]:
    ax.set_visible(False)
for ax, series in zip(axes.flat, SERIES):
    r = results[results["series"] == series].set_index("model")
    for i, model in enumerate(["baci", "before_after"]):
        if model not in r.index or pd.isna(r.loc[model, "estimate"]):
            ax.text(0, i, "too few dams", ha="center", va="center", fontsize=8, color="0.4")
            continue
        row = r.loc[model]
        est, low, high = (to_plot_scale(row, k) for k in ["estimate", "ci_low", "ci_high"])
        ax.errorbar(est, i, xerr=[[est - low], [high - est]], fmt=MODEL_STYLE[model]["marker"],
                    color=MODEL_STYLE[model]["color"], ms=8, lw=2, capsize=4)
    ax.axvline(0, color="0.5", lw=1)
    ax.set_yticks([0, 1])
    ax.set_yticklabels([f"{MODEL_LABELS[m].split(' ')[0]}\n({int(r.loc[m, 'dams']) if m in r.index else 0} dams)"
                        for m in ["baci", "before_after"]], fontsize=8)
    ax.set_ylim(-0.6, 1.6)
    log_scale = bool(r["log_scale"].dropna().iloc[0]) if r["log_scale"].notna().any() else False
    ax.set_xlabel("% change" if log_scale else r["unit"].dropna().iloc[0], fontsize=9)
    ax.set_title(series, fontsize=10)
fig.suptitle("Average removal effect across dams (95% interval)", y=1.0)
fig.tight_layout()
plt.show()

## 3. Do the dams agree?

The average can hide a lot: half the dams could go up and half down. Each dot below is **one dam's own estimated effect** (the average plus that dam's random effect). Dams with little data are pulled toward the average ("shrinkage"), so extreme dots are dams with plenty of data.

- A tight cloud on one side of 0: a consistent effect.
- A wide cloud straddling 0: dams differ. Look for what distinguishes the up dams from the down dams (dam height, distance, region) before calling it a removal effect.

The text gives the share of dams with a positive effect. Each panel is zoomed to the bulk of the dams; a note says how many extreme ones are off scale (they're in the table below the plot). `effect_sd_between_dams` in `results.csv` is the same spread as one number.

In [ ]:
rng = np.random.default_rng(0)
fig, axes = plt.subplots(2, -(-len(SERIES) // 2), figsize=(4 * -(-len(SERIES) // 2), 5.2), squeeze=False)
for ax in axes.flat[len(SERIES):]:
    ax.set_visible(False)
for ax, series in zip(axes.flat, SERIES):
    log_scale = series.split(" (")[0] in settings["log_parameters"]
    shown = []
    for i, model in enumerate(["baci", "before_after"]):
        d = dam_effects[(dam_effects["series"] == series) & (dam_effects["model"] == model)]
        if d.empty:
            continue
        x = (10 ** d["dam_effect"] - 1) * 100 if log_scale else d["dam_effect"]
        shown.append(x)
        ax.scatter(x, i + rng.uniform(-0.18, 0.18, len(d)), s=14, alpha=0.6,
                   color=MODEL_STYLE[model]["color"], marker=MODEL_STYLE[model]["marker"], linewidths=0)
        ax.text(0.98, i + 0.3, f"{(d['dam_effect'] > 0).mean():.0%} of {len(d)} > 0",
                transform=ax.get_yaxis_transform(), ha="right", fontsize=8, color="0.25")
    if shown:
        # Zoom to the bulk (3 IQRs beyond the quartiles) so one extreme dam doesn't flatten the rest.
        x = pd.concat(shown)
        q1, q3 = np.percentile(x, [25, 75])
        low, high = max(x.min(), q1 - 3 * (q3 - q1)), min(x.max(), q3 + 3 * (q3 - q1))
        pad = 0.08 * (high - low or 1)
        ax.set_xlim(min(low, 0) - pad, max(high, 0) + pad)
        if n_out := int(((x < low) | (x > high)).sum()):
            ax.text(0.02, 1.45, f"{n_out} dam(s) off scale", transform=ax.get_yaxis_transform(),
                    fontsize=7, color="0.4")
    ax.axvline(0, color="0.5", lw=1)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["BACI", "before/\nafter"], fontsize=8)
    ax.set_ylim(-0.5, 1.6)
    ax.set_xlabel("% change" if log_scale else mm.unit_label(series, False), fontsize=9)
    ax.set_title(series, fontsize=10)
fig.suptitle("Each dam's own estimated effect", y=1.0)
fig.tight_layout()
plt.show()

In [ ]:
# The dams with the largest effects (before/after) for one series, to look up by AR_ID.
d = dam_effects[(dam_effects["series"] == CHECK_SERIES) & (dam_effects["model"] == "before_after")]
pd.concat([d.nlargest(5, "dam_effect"), d.nsmallest(5, "dam_effect")]).round(3)

## 4. Can we trust the fits?

**Fit table.** `re_structure` is the random-effects structure that was fit: `dam slopes + station intercepts` is the full model; `dam slopes` means the station term had to be dropped to fit. `fit_check = check` means the optimizer reported a Hessian that isn't positive definite. That almost always comes with a random effect whose variance is estimated at 0 (e.g. dams don't differ in that respect), which doesn't bias the effect. But compare those rows with `analysis_robustness.ipynb` before quoting them. `rows_dropped_missing` counts daily values without climate data.

In [ ]:
results[["series", "model", "re_structure", "fit_check", "dams", "days", "rows_dropped_missing",
         "effect_sd_between_dams", "warnings", "error"]].fillna("")

**Residuals** (what the model doesn't explain) for `CHECK_SERIES`. What to look for:
- **left, residuals vs fitted**: a shapeless cloud around 0 is good; a funnel or curve means the scale is wrong (e.g. the parameter should be log-scaled);
- **middle, histogram**: roughly bell-shaped is fine; long tails mean a few outliers carry weight;
- **right, by group**: all four boxes centred on 0. If treated-after sits off 0, the model is missing something about the removal period.

In [ ]:
r = residuals[(residuals["series"] == CHECK_SERIES) & (residuals["model"] == "before_after")]
if r.empty:
    print(f"No residuals for {CHECK_SERIES}")
else:
    both = residuals[(residuals["series"] == CHECK_SERIES) & (residuals["model"] == "baci")]
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
    axes[0].hexbin(r["fitted"], r["residual"], gridsize=50, cmap="Blues", mincnt=1)
    axes[0].axhline(0, color="0.3", lw=1)
    axes[0].set(xlabel="fitted value", ylabel="residual", title="residuals vs fitted (before/after)")
    lim = np.nanpercentile(np.abs(r["residual"]), 99.5)
    axes[1].hist(r["residual"].clip(-lim, lim), bins=60, color="#2a78d6")
    axes[1].set(xlabel="residual", title="distribution")
    groups = (both if len(both) else r).assign(
        group=lambda g: g["role"] + " / " + np.where(g["post"] == 1, "after", "before"))
    order = sorted(groups["group"].unique())
    sns.boxplot(data=groups, x="residual", y="group", order=order, ax=axes[2], showfliers=False,
                color="#9ec5f4")
    axes[2].axvline(0, color="0.3", lw=1)
    axes[2].set(title="by group (BACI fit)" if len(both) else "by group", ylabel="")
    fig.suptitle(CHECK_SERIES)
    fig.tight_layout()
    plt.show()

## 5. How to read this, and what it can't tell you

- **Lead with BACI** where it has ~15+ dams; use before/after as supporting evidence. An effect that shows in both models, in section 3's dam cloud, *and* in `analysis_robustness.ipynb`'s BACI is the strongest claim these data support.
- **Direction expected for small-dam removals:** cooler water and more dissolved oxygen downstream (no warm, still reservoir), turbidity and phosphorus up briefly while sediment moves. Effects in the 1–5 years after removal are an average over that whole window.
- **Not in the model:** stream flow (wet vs dry years beyond rain), dams still standing between a station and the removal (not in the data), censoring (non-detects are half the detection limit), and changes in method or turbidity unit across the removal (checked in `analysis_robustness` section 3).
- **Association, not proof.** The model compares before with after, and downstream with upstream, at removed dams; it can't rule out something else that happened at the same time and place.

## 6. Changing the model

Everything a design change touches is in `SETTINGS` at the top of `analysis/mixed_model.py`; rerun the script and this notebook picks up the new outputs.

| To… | Change |
|---|---|
| model other parameters | `SETTINGS["series"]` (or `--series` on the command line) |
| change who counts as treated / control | `treated_exposure`, `control_exposure`, `exclude_former_reservoir` |
| drop flagged stations (e.g. saline) | `exclude_flags = ["qc_saline"]` |
| add or remove covariates | `covariates` (any numeric column of `analysis_measurement`) |
| require more dams | `min_dams` |
| add a term (e.g. does the effect depend on dam height?) | `model_formula()`, e.g. add `post:z_dam_height_ft` and put `dam_height_ft` in `covariates` |

A rebuilt `analysis_measurement` (new distance limits, eligibility rules, more dams) needs no code change, only a rerun. The script stops with a clear message if a column it needs (`REQUIRED_COLUMNS`) is gone.